In [106]:
import pandas as pd

df = pd.read_csv("test.csv")


In [64]:
import pandas as pd

df = pd.read_csv("test.csv")
print(f"Number of entries: {len(df):,}")

Number of entries: 5,922


In [77]:
import pandas as pd

df = pd.read_csv("test.csv")
print(f"Number of entries: {len(df):,}")

# 不去重！所有行全部保留

# 重新从 1 开始连续编号 Question_ID（每组一个新ID）
df = df.sort_values(['true_q_i', 'paraphrase_rank']).reset_index(drop=True)

# 给每个 true_q_i 分配一个从 1 开始的新 Question_ID
df['Question_ID'] = df.groupby('true_q_i').ngroup() + 1

# 每组内重新编号 paraphrase_rank 从 1 开始
df['paraphrase_rank'] = df.groupby('true_q_i').cumcount() + 1

# 转为字符串并生成标准 query_id
df['Question_ID'] = df['Question_ID'].astype(str)
df['query_id'] = df['Question_ID'] + '_p' + df['paraphrase_rank'].astype(str)

print(f"After renumbering: {len(df):,} rows → ZERO deleted")
print(f"New Question_ID range: 1 to {df['Question_ID'].nunique()}")
print(f"Each group has consecutive _p1, _p2, _p3...")

# 保存
df.to_csv("test_renumbered_from_1_no_dedup.csv", index=False)
print("Saved to: test_renumbered_from_1_no_dedup.csv")

Number of entries: 5,922
After renumbering: 5,922 rows → ZERO deleted
New Question_ID range: 1 to 1974
Each group has consecutive _p1, _p2, _p3...
Saved to: test_renumbered_from_1_no_dedup.csv


In [78]:
df.head(10)

,Question_ID,user_query,true_q_i,true_a_i,paraphrase_rank,dataset,original_id,dialogue_context,retrieved_docs,query_id
0,1,Which state or province can I attribute the cr...,"""Ew!"" is a song by a television host born where?","""Ew!"" is a song by American television host an...",1,hotpotqa,5ae68fcb5542992ae0d1635b_p1,User: Where in Greek mythology was the father ...,"[""Erika Moulet is a French journalist and tele...",1_p1
1,1,Is it correct to say Ew!'s originator lives ne...,"""Ew!"" is a song by a television host born where?","""Ew!"" is a song by American television host an...",2,hotpotqa,5ae68fcb5542992ae0d1635b_p2,User: Where in Greek mythology was the father ...,"[""Erika Moulet is a French journalist and tele...",1_p2
2,1,What city or town was Ew! written about by its...,"""Ew!"" is a song by a television host born where?","""Ew!"" is a song by American television host an...",3,hotpotqa,5ae68fcb5542992ae0d1635b_p3,User: Where in Greek mythology was the father ...,"[""Erika Moulet is a French journalist and tele...",1_p3
3,2,Do we know of any American bluegrass-cum-count...,"""Heartbreak Hurricane"" was recorded by which c...","""Heartbreak Hurricane"" was recorded by the Ame...",1,hotpotqa,5a72334755429971e9dc934c_p1,User: Who was the lead prosecutor for the soil...,"[""\""Heartbreak Hurricane\"" is a song written b...",2_p1
4,2,Did Rickie Lloyd Skaggs sing Heartbreak Hurric...,"""Heartbreak Hurricane"" was recorded by which c...","""Heartbreak Hurricane"" was recorded by the Ame...",2,hotpotqa,5a72334755429971e9dc934c_p2,User: Who was the lead prosecutor for the soil...,"[""\""Heartbreak Hurricane\"" is a song written b...",2_p2
5,2,Who uses the nickname Ricky Skaggs to identify...,"""Heartbreak Hurricane"" was recorded by which c...","""Heartbreak Hurricane"" was recorded by the Ame...",3,hotpotqa,5a72334755429971e9dc934c_p3,User: Who was the lead prosecutor for the soil...,"[""\""Heartbreak Hurricane\"" is a song written b...",2_p3
6,3,On which musical project did Axl Rose team up ...,"""Prostitute"" is a song written by Axl Rose and...",The American guitarist Robin Finck has played ...,1,hotpotqa,5a80e4b255429938b6142239_p1,User: Giselle Cossard was known as Mother Gise...,"[""\""Prostitute\"" is the fourteenth and final t...",3_p1
7,3,Whose contributions went into shaping the comp...,"""Prostitute"" is a song written by Axl Rose and...",The American guitarist Robin Finck has played ...,2,hotpotqa,5a80e4b255429938b6142239_p2,User: Giselle Cossard was known as Mother Gise...,"[""\""Prostitute\"" is the fourteenth and final t...",3_p2
8,3,What musicians wrote the lyrics and music for ...,"""Prostitute"" is a song written by Axl Rose and...",The American guitarist Robin Finck has played ...,3,hotpotqa,5a80e4b255429938b6142239_p3,User: Giselle Cossard was known as Mother Gise...,"[""\""Prostitute\"" is the fourteenth and final t...",3_p3
9,4,Who appeared on a popular quiz show as told th...,"""Twenty Two"" is an episode of the series ""The ...","""The Twilight Zone"" episode ""Twenty Two"" was a...",1,hotpotqa,5ae530e355429908b6326561_p1,User: Kam Heskin plays Paige Morgan in a 2004 ...,"[""Bennett Alfred Cerf (May 25, 1898 \u2013 Aug...",4_p1


In [79]:
# 彻底清洗：只保留完全合格的 Question_ID（每题必须同时满足以下 5 个条件）

print("开始严格清洗，只保留 100% 完美的题目...\n")

before_q = df['Question_ID'].nunique()
before_rows = len(df)

# Step 1: 必须恰好 3 条
qid_counts = df['Question_ID'].value_counts()
valid_by_count = qid_counts[qid_counts == 3].index

# Step 2: paraphrase_rank 必须正好是 {1, 2, 3}
def correct_ranks(g):
    return set(g['paraphrase_rank']) == {1, 2, 3}

valid_by_rank = df.groupby('Question_ID').filter(correct_ranks)['Question_ID'].unique()

# Step 3: 同 Question_ID 内 user_query 不能重复
def no_dup_query(g):
    return g['user_query'].duplicated().sum() == 0

valid_by_query = df.groupby('Question_ID').filter(no_dup_query)['Question_ID'].unique()

# Step 4: query_id 格式必须完全正确
df['query_id_expected'] = df['Question_ID'].astype(str) + '_p' + df['paraphrase_rank'].astype(str)
valid_by_queryid = df[df['query_id'] == df['query_id_expected']]['Question_ID'].unique()

# Step 5: 一个 Question_ID 只对应一个 true_q_i（反之亦然）
valid_by_trueqi = df.groupby('Question_ID')['true_q_i'].nunique()
valid_by_trueqi = valid_by_trueqi[valid_by_trueqi == 1].index

# 取所有条件的交集 → 绝对干净的 Question_ID
final_valid_qids = set(valid_by_count) \
    .intersection(valid_by_rank) \
    .intersection(valid_by_query) \
    .intersection(valid_by_queryid) \
    .intersection(valid_by_trueqi)

clean_df = df[df['Question_ID'].isin(final_valid_qids)].copy()

# 清理临时列
clean_df = clean_df.drop(columns=['query_id_expected'], errors='ignore')

after_q = clean_df['Question_ID'].nunique()
after_rows = len(clean_df)

print("清洗完成！")
print(f"   移除前：{before_q:,} 个题目，{before_rows:,} 条数据")
print(f"   移除后：{after_q:,} 个题目，{after_rows:,} 条数据")
print(f"   移除了 {before_q - after_q:,} 个有问题的题目（保留率 {after_q/before_q:.1%})")
print(f"   最终每题正好 3 条，总计 {after_rows:,} 条训练样本\n")

# 最终验证（必须全部通过）
assert clean_df['Question_ID'].value_counts().eq(3).all(), "仍有题目不是3条！"
assert clean_df.groupby('Question_ID')['paraphrase_rank'].apply(lambda x: set(x) == {1,2,3}).all()
assert clean_df.groupby('Question_ID')['user_query'].apply(lambda x: not x.duplicated().any()).all()
assert (clean_df['query_id'] == clean_df['Question_ID'].astype(str) + '_p' + clean_df['paraphrase_rank'].astype(str)).all()
assert clean_df.groupby('Question_ID')['true_q_i'].nunique().eq(1).all()

# 保存最终绝对干净版本
clean_df.to_csv("test_final.csv", index=False)

print("所有检查 100% 通过！")
print("数据集极度干净，可直接用于任何高要求训练！")
print("已保存 → train_paraphrased_PERFECT_CLEAN.csv")

开始严格清洗，只保留 100% 完美的题目...

清洗完成！
   移除前：1,974 个题目，5,922 条数据
   移除后：1,971 个题目，5,913 条数据
   移除了 3 个有问题的题目（保留率 99.8%)
   最终每题正好 3 条，总计 5,913 条训练样本

所有检查 100% 通过！
数据集极度干净，可直接用于任何高要求训练！
已保存 → train_paraphrased_PERFECT_CLEAN.csv


In [107]:
import pandas as pd

df = pd.read_csv("test_final.csv")
print(f"Number of entries: {len(df):,}")
# 如果你想要 lines=True 的 jsonl 格式（更常用）
df.to_json("test_final.jsonl", orient="records", lines=True, force_ascii=False)

Number of entries: 11,346


In [92]:
df.head(2)

,Question_ID,user_query,true_q_i,true_a_i,paraphrase_rank,dataset,original_id,dialogue_context,retrieved_docs,query_id
0,1,Which state or province can I attribute the cr...,"""Ew!"" is a song by a television host born where?","""Ew!"" is a song by American television host an...",1,hotpotqa,5ae68fcb5542992ae0d1635b_p1,User: Where in Greek mythology was the father ...,"[""Erika Moulet is a French journalist and tele...",1_p1
1,1,Is it correct to say Ew!'s originator lives ne...,"""Ew!"" is a song by a television host born where?","""Ew!"" is a song by American television host an...",2,hotpotqa,5ae68fcb5542992ae0d1635b_p2,User: Where in Greek mythology was the father ...,"[""Erika Moulet is a French journalist and tele...",1_p2


In [89]:
df_original = pd.read_csv("outdated/test-Copy1.csv")

In [90]:
df_original

,user_query,dialogue_context,retrieved_docs,true_q_i,true_a_i,dataset,split,original_id,turn_number
0,Which viruses may not cause prolonged inflamma...,User: When does generally MERS infection does...,"[""Title: Type I Interferon Receptor Deficiency...",Which viruses may not cause prolonged inflamma...,The viruses that may not cause prolonged infla...,covidqa,test,1421,3
1,When was the first case of COVID-19 identified?,User: How many times more likely was an infect...,"[""Title: First cases of coronavirus disease 20...",When was the first case of COVID-19 identified?,The first cases of COVID-19 were identified in...,covidqa,test,677,3
2,How many antigens could be detected by Liew's ...,User: What are examples of proinflammatory cyt...,"[""Title: Development of an ELISA-array for sim...",How many antigens could be detected by Liew's ...,Liew's multiplex ELISA test could detect 9 ant...,covidqa,test,39,3
3,What is the structure of Hantaan virus?,User: Why did the T20/N36 complex not show a t...,"[""Title: Vaccines and Therapeutics Against Han...",What is the structure of Hantaan virus?,The structure of Hantaan virus is spherical or...,covidqa,test,1468,3
4,How many people did SARS-CoV infect?,User: What does ANFIS offer? Assistant: ANFIS ...,"[""Title: Estimating the number of infections a...",How many people did SARS-CoV infect?,SARS-CoV infected 8098 reported cases and 774 ...,covidqa,test,798,3
...,...,...,...,...,...,...,...,...,...
2573,what does a medical lab scientist do,User: can you prune roses in april Assistant: ...,"[""Medical laboratory scientists, also known as...",what does a medical lab scientist do,A medical laboratory scientist conducts lab te...,msmarco,test,7274,3
2574,what is the catalyst in sodium sulfite,User: pet barn bundaberg shop hours Assistant:...,"[""Abstract. A gas-lift bubble column was used ...",what is the catalyst in sodium sulfite,The catalyst in sodium sulfite is cobalt sulfate.,msmarco,test,1422,3
2575,what causes a sore swollen eye,User: distinguishing characteristics of a temp...,"[""Red eye is caused by swollen or dilated bloo...",what causes a sore swollen eye,The swelling of the eye can be caused by traum...,msmarco,test,6499,3
2576,which region in belgium is french,User: can you prune roses in april Assistant: ...,"[""France yearns for Belgium's Wallonia region....",which region in belgium is french,The French-speaking region in Belgium is Wallo...,msmarco,test,958,3


In [111]:
df = df.merge(
    df_original[['true_q_i', 'retrieved_docs']],
    on='true_q_i',
    how='left'
)

In [113]:
df = pd.read_csv("train_final.csv")
df.head(2)

,user_query,dialogue_context,true_q_i,true_a_i,dataset,paraphrase_rank,Question_ID,query_id,query,retrieved_doc_ids,turn_number,history_qids
0,At what time is a meningitis case required by ...,User: What location within a cell is targeted ...,According to the California Code of Regulation...,A meningitis case should be reported to the Ca...,covidqa,1,136,136_p1,At what time is a meningitis case required by ...,"['D496', 'D497', 'D498', 'D499']",3.0,"[4992, 1301]"
1,What provisions of the California Code of Regu...,User: What key findings is the research pointi...,According to the California Code of Regulation...,A meningitis case should be reported to the Ca...,covidqa,2,136,136_p2,What provisions of the California Code of Regu...,"['D496', 'D497', 'D498', 'D499']",3.0,"[3138, 3050]"


In [119]:
import json

df = pd.read_csv("train_final.csv")

# 修复中文/emoji/代理对（surrogates）问题的函数
def safe_str(obj):
    return obj.encode('utf-8', 'surrogatepass').decode('utf-8', 'ignore') if isinstance(obj, str) else obj

# Step 1: 提取并去重 retrieved_docs（同时修复编码问题）
docs_list = []
doc_to_id = {}
current_id = 0

for docs_str in df['retrieved_docs_ids']:
    try:
        docs = ast.literal_eval(docs_str)
        if not isinstance(docs, list):
            docs = [docs]
    except:
        # 容错解析
        docs = str(docs_str).strip('[]').replace('""', '"').split('", "')
        docs = [d.strip(' "\'') for d in docs if d.strip(' "\'"')]

    for doc in docs:
        doc = safe_str(doc.strip())
        if doc and doc not in doc_to_id:
            doc_to_id[doc] = f"D{current_id}"
            docs_list.append({"doc_id": f"D{current_id}", "text": doc})
            current_id += 1

# 保存 D2.jsonl（强制处理 surrogates）
d2_df = pd.DataFrame(docs_list)
with open("D2_train.jsonl", "w", encoding="utf-8", errors="surrogatepass") as f:
    for record in docs_list:
        json.dump(record, f, ensure_ascii=False)
        f.write("\n")
print(f"文档库 D2 创建完成: {len(d2_df):,} 条唯一文档 → D2.jsonl")

# Step 2: 替换为 doc_id
def replace_with_ids(docs_str):
    try:
        docs = ast.literal_eval(docs_str)
        if not isinstance(docs, list):
            docs = [docs]
    except:
        docs = str(docs_str).strip('[]').replace('""', '"').split('", "')
        docs = [d.strip(' "\'') for d in docs if d.strip(' "\'"')]

    ids = []
    for doc in docs:
        doc = safe_str(doc.strip())
        ids.append(doc_to_id.get(doc, "UNKNOWN"))
    return ids

df['retrieved_doc_ids'] = df['retrieved_docs'].apply(replace_with_ids)
df = df.drop(columns=['retrieved_docs'])

# 保存训练集（同样绕过 surrogates 问题）
def to_jsonl_safe(df, path):
    with open(path, "w", encoding="utf-8", errors="surrogatepass") as f:
        for _, row in df.iterrows():
            json.dump(row.to_dict(), f, ensure_ascii=False, default=str)
            f.write("\n")

to_jsonl_safe(df, "train_paraphrased_FINAL_FOR_TRAINING.jsonl")
df.to_csv("test_final_1.csv", index=False, encoding="utf-8")

print(f"训练集更新完成！")
print(f"  总样本: {len(df):,}")
print(f"  总文档: {len(d2_df):,}")
print("文件已保存（完美绕过 surrogates 错误）：")
print("  → D2.jsonl")
print("  → train_paraphrased_FINAL_FOR_TRAINING.jsonl / .csv")
print("可直接用于训练，无编码问题！")

KeyError: 'retrieved_docs_ids'

In [116]:
# create_multiturn_ragbench_v3.py
# 正确版本：为每一条 user_query 都生成独立的对话历史 → 最终仍为 25338 条
import random
import pandas as pd

df = pd.read_json("test_final_1.jsonl",  lines=True)
print(f"原始数据加载完成: {len(df):,} 条, 来自 {df['Question_ID'].nunique():,} 个独特问题")

random.seed(42)
multiturn = []

for dataset_name, dataset_group in df.groupby('dataset'):
    # 构建 Question_ID → 所有改写行 的映射
    qid_to_rows = {}
    for qid, group in dataset_group.groupby('Question_ID'):
        qid_to_rows[qid] = group.to_dict('records')  # 每组通常 3 条

    all_qids = list(qid_to_rows.keys())

    # 遍历当前 dataset 的每一行（而不是每个 qid）
    for idx, row in dataset_group.iterrows():
        current_qid = row['Question_ID']
        current_record = row.to_dict()

        # 随机选 2 个不同的历史问题（来自同 dataset）
        available_history_qids = [q for q in all_qids if q != current_qid]
        if len(available_history_qids) < 2:
            history_qids = available_history_qids
        else:
            history_qids = random.sample(available_history_qids, k=2)

        # 构建历史对话
        history = []
        for h_qid in history_qids:
            h_ex = random.choice(qid_to_rows[h_qid])  # 从该问题随机挑一条改写
            history.append(f"User: {h_ex['user_query']} Assistant: {h_ex['true_a_i']}")

        dialogue_context = " || ".join(history)

        multiturn.append({
            **current_record,
            'dialogue_context': dialogue_context,
            'turn_number': len(history) + 1,
            'history_qids': history_qids  # 可选，用于调试
        })

# 最终结果
result_df = pd.DataFrame(multiturn)

# 保存
result_df.to_csv("test_final.csv", index=False)
result_df.to_json("test_final_2.jsonl", orient="records", lines=True, force_ascii=False)

print(f"\n成功生成 {len(result_df):,} 条多轮样本（与原始数据完全等量！）")
print(f"   每条原始 user_query 都保留并添加了独立随机对话历史")
print(f"   平均历史轮数: {result_df['turn_number'].mean():.2f}")
print(f"   样本分布与原始完全一致，训练偏差为 0")
print("   文件已保存：final_v2_correct_25338.csv / .jsonl")
print("   完美用于多轮 RAG / 长上下文 / Agent 训练！")

原始数据加载完成: 11,346 条, 来自 1,971 个独特问题

成功生成 11,346 条多轮样本（与原始数据完全等量！）
   每条原始 user_query 都保留并添加了独立随机对话历史
   平均历史轮数: 3.00
   样本分布与原始完全一致，训练偏差为 0
   文件已保存：final_v2_correct_25338.csv / .jsonl
   完美用于多轮 RAG / 长上下文 / Agent 训练！


In [1]:
# deduplicate_by_query_id.py
import json

for filename in ["train_final.jsonl", "test_final.jsonl"]:
    input_path = filename
    output_path = filename.replace(".jsonl", "_dedup.jsonl")
    
    print(f"正在处理 {filename} ...")
    
    seen_query_ids = set()
    kept_count = 0
    total_count = 0
    deduped_lines = []

    with open(input_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            total_count += 1
            
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                print(f"  第 {line_num} 行 JSON 错误，已跳过")
                continue

            query_id = obj.get("query_id")
            if not query_id:
                print(f"  第 {line_num} 行缺少 query_id，已跳过")
                continue

            if query_id not in seen_query_ids:
                seen_query_ids.add(query_id)
                deduped_lines.append(line)
                kept_count += 1
            # else: 重复的，丢弃

    # 写入去重后的文件
    with open(output_path, "w", encoding="utf-8") as f:
        for line in deduped_lines:
            f.write(line + "\n")

    print(f"  原始行数      : {total_count:,}")
    print(f"  去重后保留    : {kept_count:,}")
    print(f"  移除重复      : {total_count - kept_count:,}")
    print(f"  已保存 → {output_path}\n")

print("全部完成！去重后文件：")
print("   train_final_dedup.jsonl")
print("   test_final_dedup.jsonl")
print("每个 query_id 只保留第一次出现的记录，安全、无副作用")

正在处理 train_final.jsonl ...
  原始行数      : 29,163
  去重后保留    : 25,338
  移除重复      : 3,825
  已保存 → train_final_dedup.jsonl

正在处理 test_final.jsonl ...
  原始行数      : 7,521
  去重后保留    : 4,842
  移除重复      : 2,679
  已保存 → test_final_dedup.jsonl

全部完成！去重后文件：
   train_final_dedup.jsonl
   test_final_dedup.jsonl
每个 query_id 只保留第一次出现的记录，安全、无副作用


In [2]:
# rebuild_by_true_q_i.py
import json
from collections import defaultdict

print("正在按 true_q_i 重建 train 和 test 数据集...")

def process_file(filepath):
    print(f"\n处理 {filepath} ...")
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    data.append(json.loads(line))
                except:
                    continue

    # Step 1: 按 true_q_i 分组，收集所有 user_query
    true_qi_to_queries = defaultdict(set)
    true_qi_to_rows = defaultdict(list)

    for row in data:
        tqi = str(row.get("true_q_i", "")).strip()
        query = str(row.get("user_query", "")).strip()
        if not tqi or not query:
            continue
        true_qi_to_queries[tqi].add(query)
        true_qi_to_rows[tqi].append(row)

    print(f"  发现 {len(true_qi_to_queries)} 个唯一 true_q_i")

    # Step 2: 为每个 true_q_i 选最多 3 条不同的 user_query
    new_dataset = []
    new_qid = 0

    tqi_list = list(true_qi_to_queries.keys())
    tqi_list.sort()  # 可复现顺序

    for tqi in tqi_list:
        all_queries = list(true_qi_to_queries[tqi])
        if len(all_queries) > 3:
            # 简单策略：保留前3条（可替换为多样性采样）
            selected_queries = all_queries[:3]
        else:
            selected_queries = all_queries

        # 找到这些 query 对应的原始行（任选一条作为模板）
        template_row = None
        for row in true_qi_to_rows[tqi]:
            if str(row.get("user_query", "")).strip() in selected_queries:
                template_row = row
                break
        if not template_row:
            template_row = true_qi_to_rows[tqi][0]

        # 为每条选中的 query 生成新记录
        for rank, query in enumerate(selected_queries, 1):
            new_row = template_row.copy()
            new_row["Question_ID"] = str(new_qid)
            new_row["user_query"] = query
            new_row["query_id"] = f"{new_qid}_p{rank}"
            new_row["paraphrase_rank"] = rank
            # 可选：清理旧字段
            # new_row.pop("original_id", None)
            new_dataset.append(new_row)

        new_qid += 1

    return new_dataset

# 处理两个文件
train_new = process_file("train_final.jsonl")
test_new  = process_file("test_final.jsonl")

# 写入新文件
def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in data:
            json.dump(row, f, ensure_ascii=False)
            f.write("\n")

save_jsonl(train_new, "train_final_rebuilt.jsonl")
save_jsonl(test_new,  "test_final_rebuilt.jsonl")

print("\n" + "="*80)
print("重建完成！基于 true_q_i 的干净数据集已生成：")
print(f"   train_final_rebuilt.jsonl  → {len(train_new):,} 条")
print(f"   test_final_rebuilt.jsonl  → {len(test_new):,} 条")
print(f"   总唯一问题数（true_q_i） → {len(set(r.get('true_q_i','') for r in train_new + test_new if r.get('true_q_i'))):,}")
print("\n特性：")
print("   • 每个 true_q_i 最多 3 条不同的 user_query")
print("   • Question_ID 全新连续编号（从0开始）")
print("   • query_id 格式统一：{Question_ID}_p1/p2/p3")
print("   • 所有原始字段保留，结构标准")
print("   • 彻底解决重复、乱序、ID冲突问题")
print("="*80)
print("现在可以直接用于训练了！")

正在按 true_q_i 重建 train 和 test 数据集...

处理 train_final.jsonl ...
  发现 8803 个唯一 true_q_i

处理 test_final.jsonl ...
  发现 1614 个唯一 true_q_i

重建完成！基于 true_q_i 的干净数据集已生成：
   train_final_rebuilt.jsonl  → 26,409 条
   test_final_rebuilt.jsonl  → 4,842 条
   总唯一问题数（true_q_i） → 10,411

特性：
   • 每个 true_q_i 最多 3 条不同的 user_query
   • Question_ID 全新连续编号（从0开始）
   • query_id 格式统一：{Question_ID}_p1/p2/p3
   • 所有原始字段保留，结构标准
   • 彻底解决重复、乱序、ID冲突问题
现在可以直接用于训练了！
